**1. Смотрим на Hadoop Distributed File System**

В рамках этой части вам нужно будет обращаться к HDFS с помощью CLI, разместить файлы для следующих заданий в распределеннй файловой системе и выполнить несколько преобразований над ними.

Для работы файлы можно скачать по следующим ссылкам:
- Логи посещения сайтов юзерами за некоторый промежуток времени [ссылка](https://drive.google.com/file/d/1WXyq5WVSWwJYXPuH4kyAJ5mrR3XgfO_H/view?usp=sharing)

Разместите их в нашем внутреннем файловом хранилище с помощью HDFS CLI, для дальнейшего удобства под каждый файл стоит создать каталог с простым и понятным именем, разместить сами файлы в разных каталогах.

Набор команд, которые вам могут в этом помочь, доступны [здесь](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-common/FileSystemShell.html)

В ячейках ниже должен быть полный набор комманд ваших обращей к консоли

In [1]:
## вы можете обращаться к консоли из ноутбука таким способом
!hdfs dfs -ls /

Found 4 items
drwx------   - mapred hadoop          0 2025-04-14 14:10 /hadoop
drwxrwxrwt   - hdfs   hadoop          0 2025-04-14 14:10 /tmp
drwxrwxrwt   - hdfs   hadoop          0 2025-04-14 14:09 /user
drwxrwxrwt   - hdfs   hadoop          0 2025-04-14 14:10 /var


In [2]:
%%bash
## или же использовать для этого меджик строчку в ячейке %%bash, как вам будет удобнее

hdfs dfs -ls /

Found 4 items
drwx------   - mapred hadoop          0 2025-04-14 14:10 /hadoop
drwxrwxrwt   - hdfs   hadoop          0 2025-04-14 14:10 /tmp
drwxrwxrwt   - hdfs   hadoop          0 2025-04-14 14:09 /user
drwxrwxrwt   - hdfs   hadoop          0 2025-04-14 14:10 /var


In [33]:
!hdfs dfs -ls -R -h /user/ubuntu

drwxr-xr-x   - ubuntu hadoop          0 2025-04-14 16:04 /user/ubuntu/number1
drwxr-xr-x   - ubuntu hadoop          0 2025-04-14 16:04 /user/ubuntu/number2


In [14]:
! pwd

/home/ubuntu


In [12]:
! hdfs dfs -mkdir /home/ubuntu/number1

mkdir: `hdfs://rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/home/ubuntu': No such file or directory


In [16]:
! hdfs dfs -mkdir /home
! hdfs dfs -mkdir /home/ubuntu
! hdfs dfs -mkdir /home/ubuntu/number1
    

mkdir: Permission denied: user=ubuntu, access=WRITE, inode="/":hdfs:hadoop:drwxr-xr-x
mkdir: `hdfs://rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/home': No such file or directory
mkdir: `hdfs://rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/home/ubuntu': No such file or directory


In [18]:
! hdfs dfs -ls /user

Found 1 items
drwxr-xr-x   - hive hadoop          0 2025-04-14 14:10 /user/hive


In [20]:
! hdfs dfs -mkdir /user/ubuntu

In [23]:
! hdfs dfs -mkdir /user/ubuntu/number2

In [35]:
! hdfs dfs -put bible.txt /user/ubuntu/number1

In [45]:
! hdfs dfs -ls /user/ubuntu

Found 2 items
drwxr-xr-x   - ubuntu hadoop          0 2025-04-14 16:15 /user/ubuntu/number1
drwxr-xr-x   - ubuntu hadoop          0 2025-04-14 16:04 /user/ubuntu/number2


In [46]:
! hdfs dfs -put sites.csv /user/ubuntu/number2

In [47]:
! hdfs dfs -ls /user/ubuntu/number2

Found 1 items
-rw-r--r--   1 ubuntu hadoop   36443383 2025-04-14 16:27 /user/ubuntu/number2/sites.csv


**2. Решаем задачи MapReduce**

**2.1 Подсчет слов в тексте**

В рамках данного задания вам нужно подсчитать кол-во слов в тексте Библии (файл приложен к ДЗ в чате тг), то есть необходимо реализовать базовый функционал утилиты word count.

**Важно** - подсчитывайте число только тех слов, длина которых больше 4 символов. Проводить процесс удаления знаков препинания и прочих символов **не нужно**

Ниже вам представлены ячейки, в которых вы должны описать структуру маппера/редьсюера и ниже вызвать их в bash-скрипте запуска MR-таски

In [22]:
%%writefile mapper.py
                
import sys

for lines in sys.stdin:
    line = lines.strip()
    if line:
        words = line.split()
        for word in words:
            if len(word) > 4:
                print(word.lower() + str('\t1'))

Overwriting mapper.py


In [19]:
%%writefile reducer.py
## сюда вы пишете питоновский скрипт для работы редьюсера

import sys

word_count = 0
last_word = "dfghjkjhgfd"
for lines in sys.stdin:
    line = lines.split('\t')
    if last_word == line[0]:
        word_count += int(line[1])
    else:
        if last_word != "dfghjkjhgfd":
            print(last_word, word_count)
        last_word = line[0]
        word_count = 1
print(last_word, word_count)

Overwriting reducer.py


В качестве проверки ваших python-скриптов до запуска MR таски можно произвести их запуск через консольные команды

Тогда наша задача не будет выполняться через датаноды, а только на локальной машине, но в случае ошибок в скриптах вы увидите их и сможете исправить

In [15]:
%%bash
## пример запуска скриптов на неймноде для проверки их работы
## вывела самое большое количество использований

cat bible.txt | python3 mapper.py | sort -k1,1 | python3 reducer.py | sort -k2,2nr | head -n 50

shall 9733
which 4244
their 3859
there 2008
before 1722
lord, 1613
against 1596
shalt 1589
children 1560
said, 1556
them, 1549
saying, 1272
house 1222
every 1208
people 1194
because 1178
thee, 1170
these 1147
saith 1135
after 1128
behold, 1073
therefore 1054
israel 1025
among 907
thine 883
neither 861
great 847
brought 815
things 780
jesus 777
should 772
according 755
israel, 724
forth 720
bring 711
david 687
lord. 674
them. 674
heard 559
called 557
father 538
people, 512
spake 510
hundred 501
thee. 486
moses 481
given 476
heart 461
those 455
about 434


Как только в данной проверке вы получите успешный и корректный результат, можете запустить Map Reduce в ячейке ниже

In [16]:
%%bash
## шаблон для запуска MR таски

# обязательная чистка директории, куда будем складывать результат отрабоки mr
hdfs dfs -rm -r /user/ubuntu/number1/word_count_task || true

# запус mr таски с указанием пути до нужного jar
hadoop jar /usr/lib/hadoop-mapreduce/hadoop-streaming.jar \
    -D mapreduce.job.name="word-count" \
    -files ./mapper.py,./reducer.py \
    -mapper "python3 mapper.py" \
    -reducer "python3 reducer.py" \
    -input /user/ubuntu/number1/bible.txt \
    -output /user/ubuntu/number1/word_count_task

Deleted /user/ubuntu/number1/word_count_task
packageJobJar: [] [/usr/lib/hadoop-mapreduce/hadoop-streaming-3.2.2.jar] /tmp/streamjob7102402873053079394.jar tmpDir=null


2025-04-16 08:15:11,091 INFO client.RMProxy: Connecting to ResourceManager at rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/10.129.0.11:8032
2025-04-16 08:15:11,330 INFO client.AHSProxy: Connecting to Application History server at rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/10.129.0.11:10200
2025-04-16 08:15:11,371 INFO client.RMProxy: Connecting to ResourceManager at rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/10.129.0.11:8032
2025-04-16 08:15:11,372 INFO client.AHSProxy: Connecting to Application History server at rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/10.129.0.11:10200
2025-04-16 08:15:11,589 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/ubuntu/.staging/job_1744790413162_0007
2025-04-16 08:15:12,370 INFO mapred.FileInputFormat: Total input files to process : 1
2025-04-16 08:15:12,550 INFO mapreduce.JobSubmitter: number of splits:30
2025-04-16 08:15:12,819 INFO mapreduce.JobSubmitter: Submitting 

Мониторить процесс работы таски можно на nodemanager по порту 8088 (уже прокинут в конфиге), там будет UI, в котором будет видно вашу запущенную задачу и её статус.

Результат работы скрипта должен выглядеть следующим образом (вывод тестовый):

```bash
word count
abtr 6852
btoad 4237
stress 1932
zen 1885
```

In [17]:
%%bash
## запустите эту команду, чтобы вывести счетчик определенных слов, которые мы указали на grep
## Это нам будет необходимо для визуального анализа результата работы вашего скрипта
## в sort можете указать тот разделитель колонок, с которым у вас результат выплевывает редьюсер

hdfs dfs -cat /user/ubuntu/number1/word_count_task/* | grep  -E 'lord\.|god\.|pray\.' | sort -t$'\t' -k2.2nr  | head -n 3

lord. 674	
pray. 5	


**2.2 Решаем задачу поиска самых посещаемых сайтов**

В данном задании нужно поработать с логом данных о посещении юзерами различных сайтов.
Формат данных: `url;временная метка`. Вам нужно вывести топ 5 сайтов по посещаемости в каждую из дат, которая представлена в наших данных.

Результат работы скрипта должен выглядеть следующим образом:

```bash
date        site                            count
2024-05-25  https://gonzales-bautista.com/  987
2024-05-25  https://smith.com/              654
2024-05-25  https://www.smith.com/          321
```

**Рекомендации**

1. Вам могу пригодиться дополнительные параметры mr таски, отвечающие за настройку шаффла, и правил сортировки ключей внутри него. Почитать о примерах их использования можно [здесь](https://hadoop.apache.org/docs/current/hadoop-streaming/HadoopStreaming.html#More_Usage_Examples).

2. Не рекомендуем использовать `\t` в качестве символа разделителя для сложного ключа (потому что по дефолту таб используется для разделения колонок данных, и ключом в таком случае будет только первая колонка до таба). Если вы будете собирать сложный ключ для нужной вам сортировки данных, лучше всего будет использовать другие симловы, к примеру `+, =`.

3. Возможно, у вас не получится решить данную задачу за одну mr таску, тогда вы просто описываете в решении скрипты ваших мапперов, редьюсеров под каждую из mr тасок, которые вам нужно запустить для получения нужного результата.

**Важно** помнить, что любой маппер и редьюсер должен работать за O(1) памяти, и если вы будете создавать какой-то список, куда будете складывать какие-то данные, то он не должен быть размера O(n). Если такой момент в вашем решении будет, пожалуйста, поясните его текстово, что с вашими переменными все ок, и у них нет размера O(n).

In [183]:
%%writefile mapper1.py
import sys
import csv

reader = csv.reader(sys.stdin, delimiter=';')
for row in reader:
    date = row[1].strip()
    date = str(date)
    date = date.split()[0]
    linking = row[0].strip()
    print(f"{date} {linking}")

Overwriting mapper1.py


In [8]:
%%writefile reducer1.py

import sys

cur_date = ""
cur_link = ""
cur_count = 0
top_five = list()
for lines in sys.stdin:
    new_date, new_link = lines.split()
    if new_date == cur_date:
        if cur_link == new_link:
            cur_count += 1
        else:
            if len(top_five) >= 5:
                top_five.append([cur_link, cur_count])
                top_five = sorted(top_five, key=lambda x: x[1])
                top_five = top_five[1:]
                top_five = top_five[::-1]
            else:
                top_five.append([cur_link, cur_count])
            cur_link = new_link
            cur_count = 1
    else:
        for top_thing in top_five:
            print(cur_date, top_thing[0], top_thing[1])
        cur_date = new_date
        cur_link = new_link
        cur_count = 1
        top_five = list()

for top_thing in top_five:
    print(cur_date, top_thing[0], top_thing[1])

Overwriting reducer1.py


In [18]:
! cat sites.csv | python3 mapper1.py | sort -k1,1 | python3 reducer1.py

2024-05-26 https://gonzales-bautista.com/ 335
2024-05-26 http://smith.com/ 235
2024-05-26 https://www.smith.com/ 221
2024-05-26 http://www.smith.com/ 212
2024-05-26 https://smith.com/ 212
2024-05-27 https://gonzales-bautista.com/ 376
2024-05-27 https://www.smith.com/ 270
2024-05-27 https://smith.com/ 236
2024-05-27 http://smith.com/ 215
2024-05-27 http://www.smith.com/ 208
2024-05-28 https://gonzales-bautista.com/ 368
2024-05-28 https://smith.com/ 256
2024-05-28 https://www.smith.com/ 251
2024-05-28 http://smith.com/ 224
2024-05-28 http://www.smith.com/ 204
2024-05-29 https://gonzales-bautista.com/ 402
2024-05-29 https://www.smith.com/ 242
2024-05-29 http://www.smith.com/ 223
2024-05-29 https://smith.com/ 220
2024-05-29 http://smith.com/ 206
2024-05-30 https://gonzales-bautista.com/ 353
2024-05-30 https://smith.com/ 246
2024-05-30 https://www.smith.com/ 239
2024-05-30 http://smith.com/ 229
2024-05-30 http://www.smith.com/ 225
2024-05-31 https://gonzales-bautista.com/ 374
2024-05-31 htt

In [33]:
%%bash
## запустите эту команду, чтобы вывести результат работы по определенным компаниям, которые мы указали на grep

hdfs dfs -rm -r /user/ubuntu/number2/links_count_task || true
    
hadoop jar /usr/lib/hadoop-mapreduce/hadoop-streaming.jar \
    -D mapreduce.job.name="links-count" \
    -D mapreduce.job.reduces=1 \
    -files ./mapper1.py,./reducer1.py \
    -mapper "python3 mapper1.py" \
    -reducer "python3 reducer1.py" \
    -input /user/ubuntu/number2/sites.csv \
    -output /user/ubuntu/number2/links_count_task
    
    
##тут к вашему коду пришлось добавить -D mapreduce.job.reduces=1 \ иначе оно на несколько редьюсеров разделяет и считает много топ-5 для одной даты
##надеюсь так можно если нет то простите но других вариантов у меня нет

packageJobJar: [] [/usr/lib/hadoop-mapreduce/hadoop-streaming-3.2.2.jar] /tmp/streamjob4276349739723304002.jar tmpDir=null


rm: `/user/ubuntu/number2/links_count_task': No such file or directory
2025-04-16 08:42:55,570 INFO client.RMProxy: Connecting to ResourceManager at rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/10.129.0.11:8032
2025-04-16 08:42:55,800 INFO client.AHSProxy: Connecting to Application History server at rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/10.129.0.11:10200
2025-04-16 08:42:55,843 INFO client.RMProxy: Connecting to ResourceManager at rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/10.129.0.11:8032
2025-04-16 08:42:55,844 INFO client.AHSProxy: Connecting to Application History server at rc1b-dataproc-m-dipgdtjo0i9pko50.mdb.yandexcloud.net/10.129.0.11:10200
2025-04-16 08:42:56,071 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/ubuntu/.staging/job_1744790413162_0009
2025-04-16 08:42:56,470 INFO mapred.FileInputFormat: Total input files to process : 1
2025-04-16 08:42:56,630 INFO mapreduce.JobSubmitter: number of spl

In [34]:
!hdfs dfs -cat /user/ubuntu/number2/links_count_task/* | grep -E '2024-05-28|2024-06-02|2024-05-30' | column -t -s$'\t'

2024-05-28 https://gonzales-bautista.com/ 368
2024-05-28 https://smith.com/ 256
2024-05-28 https://www.smith.com/ 251
2024-05-28 http://smith.com/ 224
2024-05-28 http://www.smith.com/ 204
2024-05-30 https://gonzales-bautista.com/ 353
2024-05-30 https://smith.com/ 246
2024-05-30 https://www.smith.com/ 239
2024-05-30 http://smith.com/ 229
2024-05-30 http://www.smith.com/ 225
2024-06-02 http://smith.com/ 7
2024-06-02 https://gonzales-bautista.com/ 7
2024-06-02 https://www.williams.com/ 6
2024-06-02 http://lee.com/ 5
2024-06-02 https://www.jones.com/ 5
